In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from google.colab import files

# 1. DIRECTORY CONFIGURATION
dataset_dir = '/content/dataset/dataset_norwood'
IMG_SIZE = (224, 224)
BATCH_SIZE = 64

print("\n--- Loading Multi-Class Dataset ---")
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 2. CLASS WEIGHTS (Critical for Class Imbalance)
norwood_weights = {
    0: 0.49, 1: 0.70, 2: 0.68, 3: 1.20,
    4: 1.98, 5: 3.55, 6: 2.04
}

# 3. ADVANCED ARCHITECTURE (MobileNetV3Large)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomBrightness(factor=0.15)
])

# Load the Large model
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Freeze for Phase 1

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v3.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x) # Slightly higher dropout for the larger model
outputs = layers.Dense(7, activation="softmax")(x)

model = models.Model(inputs, outputs)

# 4. PHASE 1: WARM-UP (Train the Head)
print("\n--- PHASE 1: Training Classification Head ---")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=norwood_weights
)

# 5. PHASE 2: FINE-TUNING (Train the Entire Network)
print("\n--- PHASE 2: Deep Fine-Tuning ---")
base_model.trainable = True

# Freeze the absolute bottom layers, unfreeze the top 100
for layer in base_model.layers[:-100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5), # Micro learning rate
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Callbacks to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=8,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    class_weight=norwood_weights,
    callbacks=[early_stop, reduce_lr]
)

# 6. QUANTIZATION & EXPORT
print("\n--- Quantizing & Exporting ---")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model_large = converter.convert()

output_filename = "norwood_large_tuned.tflite"
with open(output_filename, "wb") as f:
    f.write(tflite_model_large)

files.download(output_filename)
print("Complete! Check your downloads.")


--- Loading Multi-Class Dataset ---
Found 5029 files belonging to 7 classes.
Using 4024 files for training.
Found 5029 files belonging to 7 classes.
Using 1005 files for validation.
12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- PHASE 1: Training Classification Head ---
Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 24s 222ms/step - accuracy: 0.3318 - loss: 1.9268 - val_accuracy: 0.5095 - val_loss: 1.2878
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.4911 - loss: 1.3761 - val_accuracy: 0.5701 - val_loss: 1.1274
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 110ms/step - accuracy: 0.5393 - loss: 1.1885 - val_accuracy: 0.6080 - val_loss: 1.0137
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 109ms/step - accuracy: 0.5649 - loss: 1.1201 - val_accuracy: 0.6527 - val_loss: 0.9387
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 112ms/step - accuracy: 0.5989 - loss: 1.0256 - val_accuracy: 0.6547 - val_loss: 0.9077
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 109ms/step - accuracy: 0.6270 - loss:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Complete! Check your downloads.
